In [ ]:
from sklearn.preprocessing import QuantileTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import PolynomialFeatures, RobustScaler
import seaborn as sns # Import sns to load dataset

# Re-load the dataset and perform feature engineering to ensure 'bill_per_head' exists.
# This ensures consistency even if previous cells were not run in order or modified.

df = sns.load_dataset('tips')
df['bill_per_head'] = df['total_bill'] / df['size'] # Correctly create the feature

# 1. Clean Feature Set - Use 'bill_per_head'
X = df[['total_bill', 'size', 'bill_per_head']]
y = df['tip']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Optimized Preprocessing
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    # QuantileTransformer is 'stronger' than Yeo-Johnson for non-normal data
    ('quantile', QuantileTransformer(output_distribution='normal', n_quantiles=100)),
    ('scaler', RobustScaler())
])

# 3. Go back to the High-Performer: HistGradientBoosting
model = TransformedTargetRegressor(
    regressor=HistGradientBoostingRegressor(
        random_state=42,
        learning_rate=0.05,
        max_iter=150,
        l2_regularization=1.0 # Subtle regularization
    ),
    transformer=QuantileTransformer(output_distribution='normal', n_quantiles=100)
)                         

final_pipe = Pipeline(steps=[
    ('preprocessor', ColumnTransformer([('num', numeric_transformer, X.columns)])),
    ('model', model)
])

final_pipe.fit(X_train, y_train)
print(f"✅ Optimized Final R2 Score: {final_pipe.score(X_test, y_test):.4f}")

✅ Optimized Final R2 Score: 0.5487